In [ ]:
import re
import random
import numpy as np
import pandas as pd
from wordfreq import top_n_list, zipf_frequency
import nltk
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from typing import List, Tuple, Dict, Optional
import joblib

# setup

In [2]:
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
N_WORDS = 100_000

vocab = top_n_list('en', N_WORDS)

def normalize(word: str) -> str:
    w = word.lower()
    w = re.sub(r"[^a-z]", "", w)
    return w

vocab = [normalize(w) for w in vocab]
vocab = [w for w in vocab if 2 <= len(w) <= 20]  # keep medium-length words
vocab = list(dict.fromkeys(vocab))  # dedupe, preserve order (common-first)
# len(vocab), vocab[:15]

In [ ]:
vocab_nostop = [w for w in vocab if w not in STOPWORDS]
# len(vocab_nostop), vocab_nostop[:15]

In [ ]:
MIN_ZIPF = 3.5   # include words used reasonably often
MAX_ZIPF = 6.5   # exclude the ultra-common function words (already removed) & super frequent terms if desired

def in_zipf_range(word: str, lo=MIN_ZIPF, hi=MAX_ZIPF) -> bool:
    z = zipf_frequency(word, 'en')
    return (z >= lo) and (z <= hi)

filtered_words = [w for w in vocab_nostop if in_zipf_range(w)]
print(len(filtered_words))

sample_size = 10
random.sample(filtered_words, sample_size)

14668


['china',
 'brands',
 'sailed',
 'nexus',
 'shouted',
 'ignoring',
 'sic',
 'senses',
 'advocates',
 'inspectors']

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL = SentenceTransformer(MODEL_NAME)

BATCH_SIZE = 1024
embeddings = []

for i in range(0, len(filtered_words), BATCH_SIZE):
    batch = filtered_words[i:i+BATCH_SIZE]
    vecs = MODEL.encode(batch, batch_size=256, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    embeddings.append(vecs)

embeddings = np.vstack(embeddings)
embeddings.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(14668, 384)

In [ ]:
EMBEDDED_VOCAB = list(zip(filtered_words, embeddings))
target_word = random.choice(EMBEDDED_VOCAB)
print(target_word[0])

prostate


# functions

In [ ]:
def _get_target_word_tuple(word: str):
    """
    Retrieve (word, embedding) pair for a given word.
    If the word is not in embedded_vocab, embed it using the model.
    """
    word = word.lower()
    for w, vec in EMBEDDED_VOCAB:
        if w == word:
            return (w, vec)

    # If not found, embed on the fly
    vec = MODEL.encode([word], convert_to_numpy=True, normalize_embeddings=True)[0]
    return (word, vec)

def words_in_similarity_range(
    target_word: str,
    low: float,
    high: float,
    top_k: int = None
) -> List[Tuple[str, float]]:
    """
    Return words whose cosine similarity to the target is within [low, high].

    Args:
        embedded_vocab: list of (word, embedding) pairs
        target_word: (word, embedding) pair for the chosen target
        low: lower bound of similarity range (inclusive)
        high: upper bound of similarity range (inclusive)
        top_k: optional, return only top_k closest by similarity (after filtering)

    Returns:
        List of (word, similarity) sorted by similarity descending
    """
    target_word_tuple = _get_target_word_tuple(target_word)
    target_vec = target_word_tuple[1]
    # Normalize once for efficiency
    target_vec = target_vec / np.linalg.norm(target_vec)

    results = []
    for word, vec in EMBEDDED_VOCAB:
        if word == target_word_tuple[0]:
            continue
        sim = float(np.dot(target_vec, vec) / (np.linalg.norm(vec) + 1e-9))
        if low <= abs(sim) <= high:  # Use absolute similarity
            results.append((word, sim))

    results.sort(key=lambda x: abs(x[1]), reverse=True) # Sort by absolute similarity
    if top_k:
        results = results[:top_k]
    return results

In [ ]:
words_in_similarity_range("risky", .45, .60)

[('risk', 0.5983454585075378),
 ('safest', 0.5970733761787415),
 ('risks', 0.5789536237716675),
 ('warned', 0.5663036704063416),
 ('safer', 0.5661525726318359),
 ('suitable', 0.5507111549377441),
 ('careful', 0.5476652383804321),
 ('dangers', 0.54466313123703),
 ('caution', 0.54434734582901),
 ('danger', 0.5420826077461243),
 ('potentially', 0.5245213508605957),
 ('possible', 0.5177586674690247),
 ('scary', 0.5175193548202515),
 ('warning', 0.5158416032791138),
 ('advise', 0.4978795051574707),
 ('afraid', 0.4977157413959503),
 ('terrifying', 0.49610787630081177),
 ('beware', 0.49043726921081543),
 ('wary', 0.48963671922683716),
 ('recommended', 0.4883454442024231),
 ('plausible', 0.48714134097099304),
 ('creative', 0.4863796830177307),
 ('worried', 0.48389068245887756),
 ('bet', 0.48357877135276794),
 ('appropriate', 0.4812871217727661),
 ('promising', 0.48072099685668945),
 ('hurry', 0.4778948724269867),
 ('unlikely', 0.47662726044654846),
 ('guaranteed', 0.4758531451225281),
 ('pract